In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
# Funciones auxiliares para mostrar los cálculos intermedios paso a paso
from itertools import combinations as _combinations


def detalle_tiempos_nodos(proyecto, duraciones):
    """Tiempos tempranos y tardíos de los nodos con los candidatos que se comparan en cada uno."""
    G = proyecto.pert_graph
    d = {data['activity']: (0 if data['activity'].startswith('@') else int(duraciones[data['activity']]))
         for _, _, data in G.edges(data=True)}
    nodos = list(proyecto.nodes)
    early, filas = {}, []
    for j in nodos:
        entrantes = [(i, data['activity']) for i, _, data in G.in_edges(j, data=True)]
        if not entrantes:
            early[j] = 0
            filas.append({'nodo': j, 'candidatos t_i + d_ij': 'nodo inicial', 'tiempo temprano = máx': 0})
            continue
        cand = {f'{a}: {early[i]} + {d[a]} = {early[i] + d[a]}': early[i] + d[a] for i, a in entrantes}
        early[j] = max(cand.values())
        filas.append({'nodo': j, 'candidatos t_i + d_ij': '; '.join(cand), 'tiempo temprano = máx': early[j]})
    tempranos = pd.DataFrame(filas).set_index('nodo')
    late, filas = {}, []
    for i in reversed(nodos):
        salientes = [(j, data['activity']) for _, j, data in G.out_edges(i, data=True)]
        if not salientes:
            late[i] = early[i]
            filas.append({'nodo': i, 'candidatos t*_j − d_ij': 'nodo final (= su tiempo temprano)', 'tiempo tardío = mín': late[i]})
            continue
        cand = {f'{a}: {late[j]} − {d[a]} = {late[j] - d[a]}': late[j] - d[a] for j, a in salientes}
        late[i] = min(cand.values())
        filas.append({'nodo': i, 'candidatos t*_j − d_ij': '; '.join(cand), 'tiempo tardío = mín': late[i]})
    tardios = pd.DataFrame(filas).set_index('nodo')
    return tempranos, tardios


def detalle_holguras(calendario, prelaciones):
    """Cómo se obtiene cada fecha y cada holgura a partir de las vecinas."""
    pred = prelaciones.immediate_predecessor
    suc = {a: sorted(s for s, ps in pred.items() if a in ps) for a in calendario.index}
    fin = int(calendario['fin_mas_temprano'].max())
    filas = []
    for a in calendario.index:
        ES, EF = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'fin_mas_temprano'])
        LS, LF = int(calendario.loc[a, 'inicio_mas_tardio']), int(calendario.loc[a, 'fin_mas_tardio'])
        dur = int(calendario.loc[a, 'duracion'])
        ps, ss = sorted(pred[a]), suc[a]
        ef_pred = {p: int(calendario.loc[p, 'fin_mas_temprano']) for p in ps}
        es_suc = {s: int(calendario.loc[s, 'inicio_mas_temprano']) for s in ss}
        lf_pred = {p: int(calendario.loc[p, 'fin_mas_tardio']) for p in ps}
        ls_suc = {s: int(calendario.loc[s, 'inicio_mas_tardio']) for s in ss}
        min_es_suc = min(es_suc.values()) if es_suc else fin
        max_lf_pred = max(lf_pred.values()) if lf_pred else 0
        filas.append({
            'actividad': a,
            'ES = máx EF pred.': (f"máx({', '.join(f'{p}: {v}' for p, v in ef_pred.items())}) = {ES}" if ps else f'inicial → {ES}'),
            'EF = ES + d': f'{ES} + {dur} = {EF}',
            'LF = mín LS suc.': (f"mín({', '.join(f'{s}: {v}' for s, v in ls_suc.items())}) = {LF}" if ss else f'final → {LF}'),
            'LS = LF − d': f'{LF} − {dur} = {LS}',
            'H_total = LS − ES': f'{LS} − {ES} = {LS - ES}',
            'H_libre = mín ES suc. − EF': f"{min_es_suc}{'' if ss else ' (fin)'} − {EF} = {min_es_suc - EF}",
            'H_indep. = máx(0, mín ES suc. − máx LF pred. − d)': f'máx(0, {min_es_suc} − {max_lf_pred} − {dur}) = {max(0, min_es_suc - max_lf_pred - dur)}',
        })
    return pd.DataFrame(filas).set_index('actividad')


def detalle_rutas(R, valores, nombre='duración', decimales=0):
    """Suma término a término de un valor (duraciones, varianzas…) a lo largo de cada ruta."""
    filas = {}
    for r in R.index:
        acts = list(R.columns[R.loc[r] == 1])
        total = float(sum(valores[a] for a in acts))
        filas[r] = {'actividades': ' - '.join(acts),
                    f'{nombre} = suma': ' + '.join(f'{a}({valores[a]:.{decimales}f})' for a in acts) + f' = {total:.{decimales}f}',
                    nombre: total}
    return pd.DataFrame(filas).T


def detalle_estimaciones(datos, a='a', m='m', b='b'):
    """Media y varianza PERT de cada actividad con los números sustituidos."""
    filas = {}
    for act, fila in datos.iterrows():
        A, M, B = float(fila[a]), float(fila[m]), float(fila[b])
        filas[act] = {'D = (a + 4m + b)/6': f'({A:g} + 4·{M:g} + {B:g})/6 = {(A + 4 * M + B) / 6:.2f}',
                      'σ² = ((b − a)/6)²': f'(({B:g} − {A:g})/6)² = {((B - A) / 6) ** 2:.3f}'}
    return pd.DataFrame(filas).T


def detalle_ackoff(proyecto, duraciones, topes, cur, pasos):
    """En cada paso de Ackoff-Sasieni: rutas críticas, opciones consideradas (actividades comunes y parejas)
    con su coste, y la elegida."""
    R = proyecto.path_matrix(dummies=False)
    d = duraciones.astype(float).copy()
    filas = []
    for paso, elegidas in sorted(pasos.items()):
        L = R @ d[R.columns]
        T = float(L.max())
        criticas = [r for r in L.index if np.isclose(L[r], T)]
        conjuntos = [set(R.columns[R.loc[r] == 1]) for r in criticas]
        con_margen = {x for x in R.columns if d[x] > topes[x] and np.isfinite(cur[x])}
        opciones = {x: float(cur[x]) for x in set.intersection(*conjuntos) & con_margen}
        if len(conjuntos) > 1:
            cand = sorted(set().union(*conjuntos) & con_margen)
            for p, q in _combinations(cand, 2):
                if all(p in s or q in s for s in conjuntos) and p not in opciones and q not in opciones:
                    opciones[f'{p}+{q}'] = float(cur[p] + cur[q])
        orden = sorted(opciones.items(), key=lambda kv: kv[1])
        filas.append({'paso': paso + 1, 'duración': int(T), 'rutas críticas': ', '.join(criticas),
                      'opciones (coste)': '; '.join(f'{k} ({v:.0f})' for k, v in orden[:6]) + (' …' if len(orden) > 6 else ''),
                      'elegida': ', '.join(elegidas), 'coste del paso': float(sum(cur[x] for x in elegidas))})
        for x in elegidas:
            d[x] -= 1
    return pd.DataFrame(filas).set_index('paso')


def detalle_valor_ganado(datos_vg, calendario, semana):
    """PV, EV y AC actividad a actividad en el punto de control."""
    filas = {}
    for a, fila in datos_vg.iterrows():
        es, dur = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'duracion'])
        frac = min(max(semana - es, 0), dur) / dur
        filas[a] = {'BAC': float(fila['BAC']), 'planificado hasta la semana': f'{min(max(semana - es, 0), dur)} de {dur}',
                    'PV = BAC·fracción': f'{fila["BAC"]:.0f}·{frac:.2f} = {fila["BAC"] * frac:.1f}',
                    'realizado (%)': int(fila['performed']),
                    'EV = BAC·%': f'{fila["BAC"]:.0f}·{fila["performed"] / 100:.2f} = {fila["BAC"] * fila["performed"] / 100:.1f}',
                    'AC': float(fila['AC']), 'PV_i': fila['BAC'] * frac, 'EV_i': fila['BAC'] * fila['performed'] / 100}
    t = pd.DataFrame(filas).T
    return t


def muestra_formula(nombre, expresion, valor, unidades=''):
    """Una línea «nombre = expresión = valor» en Markdown."""
    return Markdown(f'**{nombre}** = {expresion} = **{valor}**{(" " + unidades) if unidades else ""}')

In [3]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Considere el proyecto cuyo cuadro de prelaciones y duraciones (en semanas) se adjunta. El cliente exige que esté terminado en **28 semanas**.

In [4]:
pregunta_1 = "1. Calcule con Zaderenko los tiempos de los nodos y el calendario del proyecto. ¿Cuál es su duración? ¿Se cumple el plazo del cliente?  "
pregunta_2 = "2. Recalcule los tiempos tardíos tomando como tiempo tardío del último nodo el plazo impuesto (28) en lugar de la duración del proyecto. ¿Qué actividades tienen ahora holgura total negativa y qué significa?  "
pregunta_3 = "3. ¿Cuántas rutas del proyecto exceden el plazo y en cuánto cada una? ¿Hay alguna actividad que esté en todas ellas? ¿Cuántas semanas habría que acortarla para cumplir el plazo si fuera la única que se acorta?  "
pregunta_4 = "4. Si en lugar de acortar actividades se negociara el plazo, ¿cuál es el mínimo plazo que no obliga a acortar nada?  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. Calcule con Zaderenko los tiempos de los nodos y el calendario del proyecto. ¿Cuál es su duración? ¿Se cumple el plazo del cliente?  

2. Recalcule los tiempos tardíos tomando como tiempo tardío del último nodo el plazo impuesto (28) en lugar de la duración del proyecto. ¿Qué actividades tienen ahora holgura total negativa y qué significa?  

3. ¿Cuántas rutas del proyecto exceden el plazo y en cuánto cada una? ¿Hay alguna actividad que esté en todas ellas? ¿Cuántas semanas habría que acortarla para cumplir el plazo si fuera la única que se acorta?  

4. Si en lugar de acortar actividades se negociara el plazo, ¿cuál es el mínimo plazo que no obliga a acortar nada?  

In [5]:
#| tbl-cap: "Cuadro de datos del enunciado"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration
    ('A'     , '---'      , 4       ),
    ('B'     , '---'      , 6       ),
    ('C'     , '---'      , 1       ),
    ('D'     , 'A,B,C'    , 7       ),
    ('E'     , 'D'        , 4       ),
    ('F'     , 'D'        , 5       ),
    ('G'     , 'E,F'      , 4       ),
    ('H'     , 'G'        , 2       ),
    ('I'     , 'H'        , 7       ),
], columns=['activity', 'predecessor', 'duration']).set_index('activity')
datos_enunciado

,predecessor,duration
activity,,
A,---,4
B,---,6
C,---,1
D,"A,B,C",7
E,D,4
F,D,5
G,"E,F",4
H,G,2
I,H,7


El proyecto corresponde a la apertura de una clínica veterinaria; las actividades son las siguientes (las duraciones del cuadro están en semanas).

In [6]:
#| tbl-cap: "Descripción de las actividades"
#| label: tbl-descripciones
descripciones = pd.Series({
    'A': 'Estudio de mercado y local',
    'B': 'Proyecto de reforma y licencia',
    'C': 'Obra de reforma',
    'D': 'Instalación de quirófano y radiología',
    'E': 'Mobiliario y equipamiento clínico',
    'F': 'Software de gestión',
    'G': 'Selección de personal',
    'H': 'Acuerdos con proveedores',
    'I': 'Registro sanitario',
}, name='Descripción')
descripciones.to_frame()

,Descripción
A,Estudio de mercado y local
B,Proyecto de reforma y licencia
C,Obra de reforma
D,Instalación de quirófano y radiología
E,Mobiliario y equipamiento clínico
F,Software de gestión
G,Selección de personal
H,Acuerdos con proveedores
I,Registro sanitario


# Resolución

In [7]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
plazo = 28

## Apartado 1

In [8]:
display(Markdown(pregunta_1))

1. Calcule con Zaderenko los tiempos de los nodos y el calendario del proyecto. ¿Cuál es su duración? ¿Se cumple el plazo del cliente?  

Algoritmo de Zaderenko paso a paso, con el grafo PERT arriba y la matriz debajo. El primer paso muestra el resultado final; a partir del siguiente se construye desde el principio (avance con los botones):

In [9]:
#| tbl-cap: "Matriz de Zaderenko"
#| label: tbl-zaderenko
proyecto.display_zaderenko_tikz(datos_proyecto['duration'])

,1,2,3,4,5,6,7,8,9,10,early
1,,6.0,4.0,1.0,,,,,,,0.0
2,,,,0.0,,,,,,,6.0
3,,,,0.0,,,,,,,4.0
4,,,,,7.0,,,,,,6.0
5,,,,,,4.0,5.0,,,,13.0
6,,,,,,,0.0,,,,17.0
7,,,,,,,,4.0,,,18.0
8,,,,,,,,,2.0,,22.0
9,,,,,,,,,,7.0,24.0
10,,,,,,,,,,,31.0


In [10]:
#| tbl-cap: "Calendario del proyecto"
#| label: tbl-calendario
calendario = proyecto.calendar(datos_proyecto['duration'])
duracion_proyecto = proyecto.duration(datos_proyecto['duration'])
proyecto.display_calendario(datos_proyecto['duration'])

,inicio_mas_temprano,fin_mas_temprano,inicio_mas_tardio,fin_mas_tardio,H_total
activity,,,,,
A,0,4,2,6,2
B,0,6,0,6,0
C,0,1,5,6,5
D,6,13,6,13,0
E,13,17,14,18,1
F,13,18,13,18,0
G,18,22,18,22,0
H,22,24,22,24,0
I,24,31,24,31,0


In [11]:
Markdown(f'La duración del proyecto es {duracion_proyecto} semanas, {duracion_proyecto - plazo} más que el plazo: no se cumple.')

La duración del proyecto es 31 semanas, 3 más que el plazo: no se cumple.

## Apartado 2

In [12]:
display(Markdown(pregunta_2))

2. Recalcule los tiempos tardíos tomando como tiempo tardío del último nodo el plazo impuesto (28) en lugar de la duración del proyecto. ¿Qué actividades tienen ahora holgura total negativa y qué significa?  

Los tiempos tempranos no dependen del plazo. Los tardíos sí: si el último nodo debe alcanzarse en la semana del plazo, todos los tiempos tardíos (y los inicios y fines más tardíos de las actividades) se adelantan en la diferencia entre la duración y el plazo, y la holgura total de cada actividad disminuye en esa misma cantidad.

In [13]:
retraso = duracion_proyecto - plazo
con_plazo = calendario[['inicio_mas_temprano', 'fin_mas_temprano']].copy()
con_plazo['inicio_mas_tardio'] = calendario['inicio_mas_tardio'] - retraso
con_plazo['fin_mas_tardio'] = calendario['fin_mas_tardio'] - retraso
con_plazo['H_total'] = con_plazo['inicio_mas_tardio'] - con_plazo['inicio_mas_temprano']
con_plazo

,inicio_mas_temprano,fin_mas_temprano,inicio_mas_tardio,fin_mas_tardio,H_total
activity,,,,,
A,0,4,-1,3,-1
B,0,6,-3,3,-3
C,0,1,2,3,2
D,6,13,3,10,-3
E,13,17,11,15,-2
F,13,18,10,15,-3
G,18,22,15,19,-3
H,22,24,19,21,-3
I,24,31,21,28,-3


In [14]:
negativas = con_plazo.index[con_plazo['H_total'] < 0].tolist()
Markdown(f'''Con el plazo impuesto, {len(negativas)} actividades tienen holgura total negativa: {", ".join(negativas)}. Una holgura −h significa que, aun empezando lo antes posible, la actividad termina h semanas más tarde de lo que el plazo permite: son las actividades de las rutas que exceden el plazo (las críticas tienen −{retraso}), y hay que acortar alguna de ellas en cada una de esas rutas.''')

Con el plazo impuesto, 8 actividades tienen holgura total negativa: A, B, D, E, F, G, H, I. Una holgura −h significa que, aun empezando lo antes posible, la actividad termina h semanas más tarde de lo que el plazo permite: son las actividades de las rutas que exceden el plazo (las críticas tienen −3), y hay que acortar alguna de ellas en cada una de esas rutas.

## Apartado 3

In [15]:
display(Markdown(pregunta_3))

3. ¿Cuántas rutas del proyecto exceden el plazo y en cuánto cada una? ¿Hay alguna actividad que esté en todas ellas? ¿Cuántas semanas habría que acortarla para cumplir el plazo si fuera la única que se acorta?  

In [16]:
#| tbl-cap: "Rutas que exceden el plazo"
#| label: tbl-rutasExceden
R = proyecto.path_matrix(dummies=False)
duracion_rutas = R @ datos_proyecto['duration'][R.columns]
exceso = (duracion_rutas - plazo)
exceden = exceso[exceso > 0].sort_values(ascending=False)
pd.DataFrame({'actividades': [' - '.join(R.columns[R.loc[r] == 1]) for r in exceden.index],
              'duración': duracion_rutas[exceden.index], 'exceso sobre el plazo': exceden})

,actividades,duración,exceso sobre el plazo
Route_4,B - D - F - G - H - I,31,3
Route_3,B - D - E - G - H - I,30,2
Route_2,A - D - F - G - H - I,29,1


In [17]:
comunes = [a for a in R.columns if R.loc[exceden.index, a].all()]
if comunes:
    display(Markdown(f'Las actividades {", ".join(comunes)} están en todas las rutas que exceden el plazo. Acortando una sola de ellas '
                     f'{int(exceden.max())} semanas (el mayor exceso) se cumpliría el plazo, siempre que su duración lo permita.'))
else:
    display(Markdown('Ninguna actividad está en todas las rutas que exceden el plazo: habrá que acortar al menos dos actividades distintas.'))

Las actividades D, G, H, I están en todas las rutas que exceden el plazo. Acortando una sola de ellas 3 semanas (el mayor exceso) se cumpliría el plazo, siempre que su duración lo permita.

## Apartado 4

In [18]:
display(Markdown(pregunta_4))

4. Si en lugar de acortar actividades se negociara el plazo, ¿cuál es el mínimo plazo que no obliga a acortar nada?  

In [19]:
Markdown(f'El mínimo plazo que no obliga a acortar nada es la duración del proyecto, {duracion_proyecto} semanas: con cualquier plazo mayor o igual todas las holguras son no negativas.')

El mínimo plazo que no obliga a acortar nada es la duración del proyecto, 31 semanas: con cualquier plazo mayor o igual todas las holguras son no negativas.